In [1]:
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, CSVLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

c:\Users\sinha\anaconda3\envs\ai\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
csv_docs = []

for file in Path(r"C:\Users\sinha\Documents\Project\Ask-HR\Company Info\data").glob("*.csv"):
    loader = CSVLoader(str(file))
    csv_docs.extend(loader.load())

In [3]:
len(csv_docs)

4951

In [4]:
md_loader = DirectoryLoader(
    r"C:\Users\sinha\Documents\Project\Ask-HR\Company Info\docs",
    glob="**/*.md",
    loader_cls=TextLoader
)

md_docs = md_loader.load()

In [5]:
len(md_docs)

13

In [6]:
documents = md_docs + csv_docs

In [7]:
len(documents)

4964

In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [9]:
chunks = splitter.split_documents(documents)

In [10]:
len(chunks)

5012

In [11]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
## for building

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="PolicyVB"
)

In [13]:
retriever = vector_db.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 50})

In [30]:
ans = retriever.invoke("Scam")

In [25]:
ans[0].metadata['source'].split('\\')[-1]

'leave_policy.md'

In [31]:
for i in ans:
    print("Metadata: ",i.metadata['source'].split('\\')[-1])
    print()
    print(i.page_content)
    print()
    print("_"*50)

Metadata:  employees.csv

employee_id: E0011
full_name: Zayan Behl
gender: Female
age: 32
email: zayan.behl@fekutech.com
phone_number: +916400524278
address: H.No. 128, Pingle Nagar, Chinsurah-045053
department: AI & Machine Learning
designation: Machine Learning Engineer
reporting_manager: Alexander Chander
joining_date: 2022-10-23
employment_type: Full-time
work_location: Bhubaneswar
salary_band: 8L-18L
skills: Python, LangChain, PostgreSQL, Hugging Face
emergency_contact: Harshil Pandya

__________________________________________________
Metadata:  training_records.csv

employee_id: E0004
training: Effective Code Reviews
provider: Udemy
completed_on: 2025-06-07

__________________________________________________
Metadata:  benefits.csv

benefit: Provident Fund
details: Provident fund contributions as per Indian law.

__________________________________________________
Metadata:  reimbursement_policy.md

Submitting false, misleading, or altered expense claims is considered a serious p